## Trying to implement various chain pipelines

### First Implementing Job Description Extraction pipeline

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [1]:
from pydantic import BaseModel, Field
from typing import List, Optional

class JobDescription(BaseModel):
    title: str = Field(description="The title of the job", examples=["Software Engineer", "Data Scientist"])
    company: str = Field(description="The company offering the job", examples=["Google", "Microsoft"])
    location: str = Field(description="The location of the job", examples=["San Francisco, CA", "New York, NY"])
    education: str = Field(description="The education requirements for the job", examples=["Bachelor's degree in Computer Science", "Master's degree in Data Science"])
    experience: int = Field(description="The experience requirements for the job in years", examples=[3, 5, 7])
    employment_type: str = Field(description="The type of employment", examples=["Full-time", "Part-time", "Contract", "Internship"])
    required_skills: List[str] = Field(description="The requirements of the job", examples=["3+ years of experience in software development", "Proficiency in Python and JavaScript", "Fluent in German"])
    soft_skills: List[str] = Field(description="The soft skills required for the job", examples=["Good communication skills", "Ability to work in a team"])
    responsibilities: List[str] = Field(description="The responsibilities of the job", examples=["Develop and maintain software applications", "Collaborate with cross-functional teams"])
    salary_range: Optional[str] = Field(description="The salary range for the job", examples=["$80,000 - $120,000", "1,00,000Rs - $5,00,000Rs"])

class Resume(BaseModel):
    name: str = Field(description="The full name of the person", examples=["John Doe", "Jane Smith"])
    email: str = Field(description="The email address of the person", examples=["john.doe@example.com", "jane.smith@example.com"])
    phone: str = Field(description="The phone number of the person", examples=["123-456-7890", "987-654-3210"])
    education: List[str] = Field(description="The education details of the person", examples=["Bachelor's degree in Computer Science from XYZ University", "Master's degree in Data Science from ABC University"])
    location: str = Field(description="The location of the person", examples=["San Francisco, CA", "New York, NY"])
    experience: int = Field(description="The experience details of the person in years", examples=[3, 5, 7])
    skills: List[str] = Field(description="The skills of the person", examples=["Python, JavaScript, SQL", "Machine Learning, Data Analysis, Deep Learning"])
    soft_skills: List[str] = Field(description="The soft skills of the person", examples=["Good communication skills", "Ability to work in a team"])
    certifications: Optional[List[str]] = Field(description="The certifications of the person", examples=["AWS Certified Solutions Architect", "Certified Data Scientist"])
    languages: Optional[List[str]] = Field(description="The languages known by the person", examples=["English", "Spanish", "French"])

class MatchingResponse(BaseModel):
    score: int = Field(description="The score indicating how well the resume matches the job description", ge=0, le=100, examples=[85, 90, 95])
    matched_skills: List[str] = Field(description="The list of skills that matched between the resume and the job description", examples=["Python", "JavaScript", "Communication skills"])
    skill_gaps: List[str] = Field(description="The list of skills that are required by the job description but are missing in the resume", examples=["SQL", "Machine Learning", "Experience gaps", "Leadership experience"])
    suggestions: List[str] = Field(description="The list of suggestions to improve the resume to better match the job description", examples=["Add SQL to your skills section", "Highlight your experience with machine learning in your resume"])

class ATSResponse(BaseModel):
    score: int = Field(description="The score indicating how well the resume is optimized for ATS", ge=0, le=100, examples=[80, 85, 90])
    issues: List[str] = Field(description="The list of issues that are affecting the resume's ATS optimization", examples=["Missing keywords", "Unusual formatting", "Lack of section headers"])
    suggestions: List[str] = Field(description="The list of suggestions to improve the resume's ATS optimization", examples=["Add relevant keywords from the job description", "Use standard section headers like 'Experience' and 'Education'", "Avoid using tables and graphics in your resume"])

In [2]:
JD_PARSER_PROMPT = """
You are a helpful assistant that extracts relevant information from a job description.
The information you extract will be used to analyze a resume and generate a cover letter.
You must focus on information that can be used to check how well the a resume matches a job description and to generate a cover letter that is tailored to the job description.
The information you extract should be in the form of a JSON object.
"""

RESUME_PARSER_PROMPT = """
You are a helpful assistant that extracts relevant information from a resume.
The information you extract will be used to analyze a job description and generate a cover letter.
You must focus on information that can be used to check how well the resume matches a job description and to generate a cover letter that is tailored to the job description.
You must also extract the person's name and contact, skills and experience etc.
The information you extract should be in the form of a JSON object.
"""

MATCH_ANALYZER_PROMPT = """
You are a helpful assistant that analyzes how well a resume matches a job description.
The information you analyze will be used to generate a cover letter that is tailored to the job description.
The information you analyze should be in the form of a JSON object.
The JSON object should contain a score that indicates how well the resume matches the job description
The information must include the list of skill gaps, matched skills, and experience gaps.
"""

ATS_CHECKER_PROMPT = """
You are a helpful assistant that checks how well a resume is optimized for Applicant Tracking Systems (ATS).
The information you analyze will be used to generate a cover letter that is tailored to the job description
The information you analyze should be in the form of a JSON object.
The JSON object should contain a score that indicates how well the resume is optimized for ATS.
"""

COVER_LETTER_GENERATOR_PROMPT = """
You are a helpful assistant that generates a cover letter based on a job description and a resume.
The cover letter should be tailored to the job description and should highlight the relevant skills and experience from the resume.
Make appropriate decisions about information that are missing in the resume but are relevant to the job description
The cover letter should be in the form of a well-written text that can be sent to a potential employer.
The cover letter should be formal and should follow the standard format of a cover letter, including an introduction, body, and conclusion.
For writing the date section use the format of day-th month year, for example, 1st January 2024, 10th March 2024 etc.
There should be proper subject line in the cover letter. Don't use things like RE. eg. "Subject: Application for Software Engineer position at Google"
"""

COVER_EMAIL_GENERATOR_PROMPT = """
You are a helpful assistant that generates a cover email based on a job description and a resume.
The cover email should be tailored to the job description and should highlight the relevant skills and experience from the resume.
Make appropriate decisions about information that are missing in the resume but are relevant to the job description
"""


In [21]:
import os, requests
r = requests.get(
  "https://generativelanguage.googleapis.com/v1beta/models",
  params={"key": os.environ["GOOGLE_API_KEY"]}
)

for model in r.json()["models"]:
    print(model["name"])

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/gemini-embedding-2-

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI

google_api = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model = "models/gemini-3-flash-preview",
    api_key = google_api
)

llm.invoke("Hello")



AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EpwDCpkDAb4+9vuAfF16+rjFbaQCM1vr35UPCJDrbv8ZsQ0+TghHHHMA7ksqjx47aEvfnywzF3YYaKMKt6SOc9frE7B9+1BzK2C6gJ9P0vDBVCPObPwBD6vR5C1ctRyIhajtZYpX5K37dJSBWD53fCAXy3geYE72yS/V1m99y9pHqDpFtGW6JZNGDyM2j08yrR5OE4cfnpSlcdexvpMNn3neIynsxAsgh1tt24oA0QOSMHCAgfsr3n9IRu05+kV9vJ/1juP9t8rrhK2dh059EoWRURSsg3x2/gRQsM2u2p5ld7ftX33K5hcuYp0CrYXTy3K7m2ae9c3SzzwS0g76V/wW95fm7PGtWU4/oxdSOGfAj/P89LmcCsMV+gMX6XDy5SxAI3X2iwmpy84sxo37U6PcSuQRM1OOEWt/oGoi2Y3pBvSpM+uCR1hmeATUS3Dkm1kQKIjMLv2NC0DNLdH1l9ZFQgxJMeMlyUj34jUVdb3cugLEYLsqGlG2FFB+pGlqjZExiaD908U8TMXqUcpqCm1BamwpGYSwK+U3/IiBMQ=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d0fc2-8201-7661-ba41-af8df014e417-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 99, 'total_tokens': 101, 'i

In [6]:
job_desc = """
**Job Title:** Senior Cloud Integration Specialist
**Department:** Technology & Innovation
**Reports To:** Director of Platform Engineering
**Location:** Austin, Texas (Hybrid - 3 days in-office)

**About the Company:**
At **NexusLogic Solutions**, we're not just riding the wave of digital transformation—we're building the surfboard. We partner with forward-thinking enterprises to streamline their operations through bespoke cloud architectures and intelligent automation. Our culture is built on curiosity, collaboration, and a healthy dose of caffeine. We were recently named one of the "Best Places to Work" by *Austin Business Journal* for the third year running.

**Position Overview:**
We are looking for a hands-on **Senior Cloud Integration Specialist** to join our growing Platform Engineering team. In this role, you will be the bridge between our development teams and cloud infrastructure, designing robust APIs and integrating third-party services. You will take ownership of our middleware solutions, ensuring data flows seamlessly and securely between our legacy systems and modern microservices.

**Key Responsibilities:**

- **Architect & Implement:** Design and develop scalable integration solutions using AWS services (API Gateway, Lambda, SQS) and integration frameworks like MuleSoft or Apache Camel.
- **API Management:** Lead the full lifecycle of API development, from design and documentation (OpenAPI/Swagger) to deployment and versioning.
- **Troubleshooting:** Act as the escalation point for complex integration issues, diagnosing bottlenecks and resolving data inconsistencies in production environments.
- **Collaboration:** Work closely with Software Engineers, Data Scientists, and Product Managers to translate business requirements into technical integration specs.
- **Mentorship:** Guide junior developers on best practices for integration patterns, security protocols (OAuth 2.0, JWT), and code maintainability.

**Qualifications:**

- 5+ years of experience in software development or integration engineering.
- Deep expertise in at least one major cloud provider (AWS preferred).
- Strong proficiency in Java, Python, or Node.js.
- Experience with message queues and streaming platforms (Kafka, RabbitMQ).
- Bachelor's degree in Computer Science or equivalent practical experience.

**Why Join NexusLogic?**

- **Compensation:** Competitive salary range: **$145,000 - $175,000** per year, plus performance-based bonus.
- **Benefits:** Comprehensive medical, dental, and vision coverage; 401(k) with 5% company match.
- **Work-Life Balance:** Generous PTO policy, paid parental leave, and flexible working hours.
- **Perks:** Weekly team lunches, annual innovation retreat, and a $1,500 annual stipend for professional development.

---

**How to Apply:**
Please send your resume and a brief cover letter to **Hiring Manager, Alex Chen**, at **careers@nexuslogic-solutions.io** with the subject line: "Senior Cloud Integration Specialist Application"."""

In [7]:
from langchain.messages import HumanMessage
from langchain.agents import create_agent

def parse_job_description(llm, job_desc: str) -> JobDescription:
    agent = create_agent(
        model = llm,
        system_prompt = JD_PARSER_PROMPT,
        response_format = JobDescription
    )
    query = HumanMessage(content = job_desc)
    response = agent.invoke({
        "messages": [query]
    })
    return response['structured_response']

In [25]:
job_description = parse_job_description(llm, job_desc)
#.model_dump_json(indent=2)

print(job_description)

title='Senior Cloud Integration Specialist' company='NexusLogic Solutions' location='Austin, Texas (Hybrid - 3 days in-office)' education="Bachelor's degree in Computer Science or equivalent practical experience" experience=5 employment_type='Full-time' required_skills=['AWS', 'API Gateway', 'Lambda', 'SQS', 'MuleSoft', 'Apache Camel', 'API development', 'OpenAPI', 'Swagger', 'Java', 'Python', 'Node.js', 'Kafka', 'RabbitMQ', 'OAuth 2.0', 'JWT'] soft_skills=['Collaboration', 'Mentorship', 'Troubleshooting', 'Problem-solving', 'Communication'] responsibilities=['Design and develop scalable integration solutions using AWS services', 'Lead the full lifecycle of API development, from design to deployment', 'Act as the escalation point for complex integration issues and data inconsistencies', 'Translate business requirements into technical integration specifications', 'Guide junior developers on best practices for integration patterns and security'] salary_range='$145,000 - $175,000'


In [26]:
print(job_description.model_dump_json(indent=2))

{
  "title": "Senior Cloud Integration Specialist",
  "company": "NexusLogic Solutions",
  "location": "Austin, Texas (Hybrid - 3 days in-office)",
  "education": "Bachelor's degree in Computer Science or equivalent practical experience",
  "experience": 5,
  "employment_type": "Full-time",
  "required_skills": [
    "AWS",
    "API Gateway",
    "Lambda",
    "SQS",
    "MuleSoft",
    "Apache Camel",
    "API development",
    "OpenAPI",
    "Swagger",
    "Java",
    "Python",
    "Node.js",
    "Kafka",
    "RabbitMQ",
    "OAuth 2.0",
    "JWT"
  ],
  "soft_skills": [
    "Collaboration",
    "Mentorship",
    "Troubleshooting",
    "Problem-solving",
    "Communication"
  ],
  "responsibilities": [
    "Design and develop scalable integration solutions using AWS services",
    "Lead the full lifecycle of API development, from design to deployment",
    "Act as the escalation point for complex integration issues and data inconsistencies",
    "Translate business requirements into 

## Resume Parsing

1. Read PDF and convert to string
2. Parse the resume

In [8]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "resume.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()
print(docs)

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-16T20:10:34+05:30', 'author': 'MIDHUN U', 'moddate': '2026-03-16T20:10:34+05:30', 'source': 'resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='AMAL VARGHESE \nLinkedIn | GitHub | Portfolio amalvarghesealiyattukudy.mec@gmail.com \nDOB – 15/05/2004 +91 9207506741 \nGovt. Model Engineering College, Kochi \nSKILLS & INTERESTS \n• Technical Skills: Python, Java, MySQL, MongoDB, Data Structures and Algorithms, OOPS, React, React Native \n• Soft Skills:  Communication, Problem Solving, Teamwork, Adaptability, Critical Thinking, Attention to Detail, Research, \n                                                 Time Management, Team Collaboration                                                                  \n• Interests:  Machine Learning, Web Development, Artificial Intelligence \nEDUCATION \n• Govt. Model Engineering College

In [9]:
print(docs[0].page_content)

AMAL VARGHESE 
LinkedIn | GitHub | Portfolio amalvarghesealiyattukudy.mec@gmail.com 
DOB – 15/05/2004 +91 9207506741 
Govt. Model Engineering College, Kochi 
SKILLS & INTERESTS 
• Technical Skills: Python, Java, MySQL, MongoDB, Data Structures and Algorithms, OOPS, React, React Native 
• Soft Skills:  Communication, Problem Solving, Teamwork, Adaptability, Critical Thinking, Attention to Detail, Research, 
                                                 Time Management, Team Collaboration                                                                  
• Interests:  Machine Learning, Web Development, Artificial Intelligence 
EDUCATION 
• Govt. Model Engineering College 9.37 | 2027 
KTU, B.Tech in Computer Science Engineering  
• Greenvalley Public School 94.4% | 2022 
CBSE, 12th     
• St. Thomas Public School 95% | 2020 
CBSE, 10th  
WORK EXPERIENCE 
• Wrench Solutions 1 Month, Ongoing 
Intern 
Technologies Used: Python, Pandas, NumPy, Matplotlib, PyTorch 
Currently working as a Mac

In [10]:
RESUME_PARSER_PROMPT = """
You are a helpful assistant that extracts relevant information from a resume.
The information you extract will be used to analyze a job description and generate a cover letter.
You must focus on information that can be used to check how well the resume matches a job description and to generate a cover letter that is tailored to the job description.
You must also extract the person's name and contact, skills and experience etc.
The information you extract should be in the form of a JSON object.
"""

class Resume(BaseModel):
    name: str = Field(description="The full name of the person", examples=["John Doe", "Jane Smith"])
    email: str = Field(description="The email address of the person", examples=["john.doe@example.com", "jane.smith@example.com"])
    phone: str = Field(description="The phone number of the person", examples=["123-456-7890", "987-654-3210"])
    education: List[str] = Field(description="The education details of the person", examples=["Bachelor's degree in Computer Science from XYZ University", "Master's degree in Data Science from ABC University"])
    location: str = Field(description="The location of the person", examples=["San Francisco, CA", "New York, NY"])
    experience: int = Field(description="The experience details of the person in years", examples=[3, 5, 7])
    skills: List[str] = Field(description="The skills of the person", examples=["Python, JavaScript, SQL", "Machine Learning, Data Analysis, Deep Learning"])
    soft_skills: List[str] = Field(description="The soft skills of the person", examples=["Good communication skills", "Ability to work in a team"])
    certifications: Optional[List[str]] = Field(description="The certifications of the person", examples=["AWS Certified Solutions Architect", "Certified Data Scientist"])
    languages: Optional[List[str]] = Field(description="The languages known by the person", examples=["English", "Spanish", "French"])

In [11]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

def parse_resume(llm: ChatGoogleGenerativeAI, resume_text: str) -> Resume:
    agent = create_agent(
        model = llm,
        system_prompt = RESUME_PARSER_PROMPT,
        response_format = Resume
    )
    query = HumanMessage(content = resume_text)
    response = agent.invoke({
        "messages": [query]
    })
    return response['structured_response']


In [31]:
resume = parse_resume(llm, docs[0].page_content)

In [32]:
print(resume.model_dump_json(indent=2))

{
  "name": "AMAL VARGHESE",
  "email": "amalvarghesealiyattukudy.mec@gmail.com",
  "phone": "+91 9207506741",
  "education": [
    "B.Tech in Computer Science Engineering, Govt. Model Engineering College (2027)",
    "12th CBSE, Greenvalley Public School (2022)",
    "10th CBSE, St. Thomas Public School (2020)"
  ],
  "location": "Kochi",
  "experience": 0,
  "skills": [
    "Python",
    "Java",
    "MySQL",
    "MongoDB",
    "Data Structures and Algorithms",
    "OOPS",
    "React",
    "React Native",
    "Pandas",
    "NumPy",
    "Matplotlib",
    "PyTorch",
    "ONNX",
    "FastAPI",
    "Next.js",
    "LangChain",
    "BrightData",
    "SQL",
    "XGBoost",
    "Supabase"
  ],
  "soft_skills": [
    "Communication",
    "Problem Solving",
    "Teamwork",
    "Adaptability",
    "Critical Thinking",
    "Attention to Detail",
    "Research",
    "Time Management",
    "Team Collaboration"
  ],
  "certifications": [
    "Programming, Data Structures and Algorithms using Python (

## Agent for Analyzing resume with Job description

In [25]:
def analyze_match(llm, job_desc: JobDescription, resume: Resume) -> MatchingResponse:
    """
    This agent will take the job description and resume in json format and analyze how well the resume matches the job description.
    
    Args:
        llm: The language model instance to use for analysis.
        job_desc (JobDescription)
        resume (Resume)
    Return:
        MatchingResponse: A JSON object containing the match percentage and a list of matching and non-matching skills, experience, education etc.
    """
    agent = create_agent(
        model = llm,
        system_prompt = MATCH_ANALYZER_PROMPT,
        response_format = MatchingResponse
    )

    query1 = HumanMessage(content = job_desc.model_dump_json(indent=2))
    query2 = HumanMessage(content = resume.model_dump_json(indent=2))
    response = agent.invoke({
        "messages": [query1, query2]
    })

    return response['structured_response']

In [34]:
response = analyze_match(llm, job_description, resume)

In [35]:
print(response.model_dump_json(indent=2))

{
  "score": 15,
  "matched_skills": [
    "Java",
    "Python",
    "Communication",
    "Problem-solving",
    "Collaboration"
  ],
  "skill_gaps": [
    "AWS",
    "API Gateway",
    "Lambda",
    "SQS",
    "MuleSoft",
    "Apache Camel",
    "OpenAPI",
    "Swagger",
    "Node.js",
    "Kafka",
    "RabbitMQ",
    "OAuth 2.0",
    "JWT"
  ],
  "suggestions": [
    "Acquire 5 years of professional experience as this is a Senior-level role and you are currently a student graduating in 2027",
    "Develop a deep understanding of AWS cloud infrastructure specifically focusing on Lambda, API Gateway, and SQS",
    "Learn enterprise integration frameworks such as MuleSoft or Apache Camel which are core requirements for this position",
    "Implement industry-standard security protocols like OAuth 2.0 and JWT in your backend projects",
    "Gain exposure to message brokers and streaming platforms like Kafka or RabbitMQ",
    "Work on projects involving API documentation standards using O

## Agent For checking ATS Friendliness

In [26]:
def check_ats(llm, resume: str) -> ATSResponse:
    """
    This agent will take the resume in string format and analyze how well it is optimized for ATS.
    Args:
        llm: The language model instance to use for analysis.
        resume (string): The resume text to analyze.
    Return:
        ATSResponse: a json response containing ATS score and suggestions
    """
    agent = create_agent(
        model = llm,
        system_prompt = ATS_CHECKER_PROMPT,
        response_format = ATSResponse
    )
    query = HumanMessage(content = resume)

    response = agent.invoke({
        "messages": [query]
    })

    return response['structured_response']

In [16]:
resume_text = docs[0].page_content
ats_report = check_ats(llm, resume_text)

NameError: name 'check_ats' is not defined

In [38]:
print(ats_report.model_dump_json(indent=2))

{
  "score": 85,
  "issues": [
    "The inclusion of Date of Birth (DOB) is unnecessary and can lead to unconscious bias or privacy issues.",
    "The Hobbies section contains non-professional information that does not contribute to the ATS score.",
    "The References section is generally not required on a resume unless specifically requested and takes up valuable space.",
    "The date formatting for the current internship ('1 Month, Ongoing') is non-standard for most ATS parsers.",
    "The use of pipes and special characters in the header (e.g., '|', '+91') can occasionally cause parsing errors in older systems."
  ],
  "suggestions": [
    "Remove the Date of Birth and Hobbies sections to keep the document strictly professional.",
    "Replace the 'References' section with a 'Certifications' or 'Volunteer Experience' section, or expand on existing project details.",
    "Standardize dates to a 'Month Year – Month Year' or 'Month Year – Present' format (e.g., 'June 2024 – Present')

### Agent to build cover letter generator

In [39]:
def generate_cover_letter(llm, job_desc: JobDescription, resume: Resume, match_response: MatchingResponse, tone: str = "Professional") -> str:
    """
    This agent will take the job description and resume in json format and the match response and generate a cover letter that is tailored to the job description and highlights the skills and experience that match the job description.
    Args:
        llm: The language model instance to use for analysis.
        job_desc (JobDescription)
        resume (Resume)
        match_response (MatchingResponse)
    Return:
        str: A cover letter that is tailored to the job description and highlights the skills and experience that match the job description.
    """

    agent = create_agent(
        model = llm,
        system_prompt = COVER_LETTER_GENERATOR_PROMPT
    )

    query1 = HumanMessage(content=job_desc.model_dump_json(indent=2))
    query2 = HumanMessage(content = resume.model_dump_json(indent=2))
    query3 = HumanMessage(content = match_response.model_dump_json(indent=2))
    query4 = HumanMessage(content = f"The tone of the cover letter should be {tone}")

    response = agent.invoke({
        "messages": [query1, query2, query3, query4]
    })

    return response["messages"][4].content[0]["text"]

In [40]:
letter = generate_cover_letter(llm, job_description, resume, response)

In [41]:
print(letter)

Amal Varghese
Kochi, Kerala
+91 9207506741
amalvarghesealiyattukudy.mec@gmail.com

24th May 2024

Hiring Manager
NexusLogic Solutions
Austin, Texas

Subject: Application for Senior Cloud Integration Specialist position at NexusLogic Solutions

Dear Hiring Manager,

I am writing to express my strong interest in the Senior Cloud Integration Specialist position at NexusLogic Solutions, as advertised. With a robust foundation in computer science engineering and a deep technical proficiency in Java and Python, I am eager to contribute my skills in API development and scalable system design to your innovative team in Austin.

Throughout my academic career and technical projects at Govt. Model Engineering College, I have developed a keen interest in building high-performance, distributed systems. My experience with frameworks such as FastAPI and Next.js has provided me with a comprehensive understanding of the full API development lifecycle. I have successfully designed and deployed backend a

In [42]:
def generate_cover_email(llm, job_desc: JobDescription, resume: Resume, match_response: MatchingResponse) -> str:
    """
    This agent will take the job description and resume in json format and the match response and generate a cover letter that is tailored to the job description and highlights the skills and experience that match the job description.
    Args:
        llm: The language model instance to use for analysis.
        job_desc (JobDescription)
        resume (Resume)
        match_response (MatchingResponse)
    Return:
        str: A cover email that is tailored to the job description and highlights the skills and experience that match the job description.
    """
    agent = create_agent(
        model = llm,
        system_prompt = COVER_EMAIL_GENERATOR_PROMPT
    )

    query1 = HumanMessage(content=job_desc.model_dump_json(indent=2))
    query2 = HumanMessage(content = resume.model_dump_json(indent=2))
    query3 = HumanMessage(content = match_response.model_dump_json(indent=2))

    response = agent.invoke({
        "messages": [query1, query2, query3]
    })

    return response["messages"][3].content[0]["text"]

In [43]:
email = generate_cover_email(llm, job_description, resume, response)

In [44]:
print(email)

Subject: Application for Senior Cloud Integration Specialist - Amal Varghese

Dear Hiring Manager,

I am writing to express my strong interest in the Senior Cloud Integration Specialist position at NexusLogic Solutions. With a robust technical foundation in Java and Python, combined with extensive hands-on experience in building scalable backend architectures and API-driven applications, I am eager to contribute to the high-level integration solutions your team is developing in Austin.

Throughout my technical tenure at Govt. Model Engineering College, I have focused on mastering the core languages required for this role—specifically Java and Python. My experience developing high-performance backends with FastAPI has provided me with a deep understanding of the API lifecycle, from design to deployment. I am well-versed in the principles of OpenAPI and Swagger for documentation, ensuring that technical specifications are clear and actionable for cross-functional teams.

Key highlights o

## Trying in Open Router

In [12]:
import os
from dotenv import load_dotenv

load_dotenv()

open_router_api = os.getenv("OPENROUTER_API_KEY")

In [13]:
#"anthropic/claude-sonnet-4.5"
#"openai/gpt-4o"

from langchain_openrouter import ChatOpenRouter
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage

model = ChatOpenRouter(
    model = "openai/gpt-4o",
    temperature = 0,
    max_tokens = 1024,
    max_retries = 2
)

test_agent = create_agent(
    model = model
)

response = test_agent.invoke({
    "messages": [
        HumanMessage(content="What is the capital of moon?"),
        AIMessage(content="The capital of the moon is Luna City."),
        HumanMessage(content="What is the population of Luna City?")
        ]
})


In [14]:
response

{'messages': [HumanMessage(content='What is the capital of moon?', additional_kwargs={}, response_metadata={}, id='9b09af82-1cd4-4b25-9812-fc0946ed3e61'),
  AIMessage(content='The capital of the moon is Luna City.', additional_kwargs={}, response_metadata={}, id='ac4bff48-f29c-4cb7-817b-3be506199766', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='What is the population of Luna City?', additional_kwargs={}, response_metadata={}, id='8b8e17a4-9091-4c14-8312-985db3a65b7b'),
  AIMessage(content='Luna City is a fictional place and does not exist in reality. Therefore, it does not have a population. The moon does not have any permanent human settlements or a capital. Any references to cities or populations on the moon are purely speculative or part of science fiction.', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-4o', 'id': 'gen-1774085958-AKhbtbSbrJgbLly64n1u', 'created': 1774085958, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': No

## Trying to make Parallel Chains

In [48]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

parse_jd_runnable = RunnableLambda(lambda x: parse_job_description(llm, x["job_desc"])) # type: ignore
parse_resume_runnable = RunnableLambda(lambda x: parse_resume(llm, x["resume"])) # type: ignore

match_analysis_runnable = RunnableParallel({
    "parsed_jd": parse_jd_runnable,
    "parsed_resume": parse_resume_runnable
})

* Ran with ChatGenerativeAI

In [51]:
data = {
    "job_desc": job_desc,
    "resume": resume_text
}

match_analysis_response = match_analysis_runnable.invoke(data)

In [55]:
print(match_analysis_response["parsed_jd"].model_dump_json(indent = 2))

{
  "title": "Senior Cloud Integration Specialist",
  "company": "NexusLogic Solutions",
  "location": "Austin, Texas (Hybrid - 3 days in-office)",
  "education": "Bachelor's degree in Computer Science or equivalent practical experience",
  "experience": 5,
  "employment_type": "Full-time",
  "required_skills": [
    "AWS (API Gateway, Lambda, SQS)",
    "Integration frameworks (MuleSoft, Apache Camel)",
    "Java",
    "Python",
    "Node.js",
    "API Management (OpenAPI/Swagger)",
    "Message queues and streaming platforms (Kafka, RabbitMQ)",
    "Security protocols (OAuth 2.0, JWT)",
    "Microservices",
    "Middleware solutions"
  ],
  "soft_skills": [
    "Collaboration",
    "Mentorship",
    "Troubleshooting",
    "Problem-solving",
    "Communication"
  ],
  "responsibilities": [
    "Design and develop scalable integration solutions using AWS services and integration frameworks like MuleSoft or Apache Camel",
    "Lead the full lifecycle of API development from design and d

In [56]:
print(match_analysis_response["parsed_resume"].model_dump_json(indent = 2))

{
  "name": "AMAL VARGHESE",
  "email": "amalvarghesealiyattukudy.mec@gmail.com",
  "phone": "+91 9207506741",
  "education": [
    "Govt. Model Engineering College, KTU, B.Tech in Computer Science Engineering (Expected 2027)",
    "Greenvalley Public School, CBSE, 12th (2022)",
    "St. Thomas Public School, CBSE, 10th (2020)"
  ],
  "location": "Kochi",
  "experience": 0,
  "skills": [
    "Python",
    "Java",
    "MySQL",
    "MongoDB",
    "Data Structures and Algorithms",
    "OOPS",
    "React",
    "React Native",
    "Pandas",
    "NumPy",
    "Matplotlib",
    "PyTorch",
    "ONNX",
    "FastAPI",
    "Next.js",
    "LangChain",
    "BrightData",
    "SQL",
    "XGBoost"
  ],
  "soft_skills": [
    "Communication",
    "Problem Solving",
    "Teamwork",
    "Adaptability",
    "Critical Thinking",
    "Attention to Detail",
    "Research",
    "Time Management",
    "Team Collaboration"
  ],
  "certifications": [
    "Programming, Data Structures and Algorithms using Python -

In [20]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

def parsing_in_parallel(llm, job_desc: str, resume_text: str):
    parse_jd_runnable = RunnableLambda(
        lambda x: parse_job_description(llm, x["job_desc"])#type: ignore
    )
    parse_resume_runnable = RunnableLambda(
        lambda x: parse_resume(llm, x["resume"])#type: ignore
    )

    # 2. Bundle them into a Parallel runner
    parsing_step = RunnableParallel({
        "parsed_jd": parse_jd_runnable,
        "parsed_resume": parse_resume_runnable,
        "raw_resume": RunnableLambda(lambda x: x["resume"]) # type: ignore
    })

    # 3. Invoke with the initial dictionary
    response = parsing_step.invoke({
        "job_desc": job_desc,
        "resume": resume_text
    })
    
    return response

In [21]:
parsing_response2 = parsing_in_parallel(model, job_desc, resume_text)

In [22]:
from pprint import pprint

pprint(parsing_response2)

{'parsed_jd': JobDescription(title='Senior Cloud Integration Specialist', company='NexusLogic Solutions', location='Austin, Texas (Hybrid - 3 days in-office)', education="Bachelor's degree in Computer Science or equivalent practical experience", experience=5, employment_type='Full-time', required_skills=['AWS services (API Gateway, Lambda, SQS)', 'Integration frameworks (MuleSoft, Apache Camel)', 'API development (OpenAPI/Swagger)', 'Java', 'Python', 'Node.js', 'Message queues and streaming platforms (Kafka, RabbitMQ)', 'Security protocols (OAuth 2.0, JWT)'], soft_skills=['Collaboration', 'Mentorship', 'Troubleshooting'], responsibilities=['Design and develop scalable integration solutions using AWS services and integration frameworks', 'Lead the full lifecycle of API development', 'Act as the escalation point for complex integration issues', 'Work closely with Software Engineers, Data Scientists, and Product Managers', 'Guide junior developers on best practices'], salary_range='$145,0

### parallel chain for analysis

In [23]:
def analysis_in_parallel(llm, job_desc: JobDescription, resume: Resume, raw_resume: str):
    matching_runnable = RunnableParallel({
        "matching_analysis": RunnableLambda(lambda x: analyze_match(llm, x["job_desc"], x["resume"])), # type: ignore
        "ats_result": RunnableLambda(lambda x: check_ats(llm, x["raw_resume"])) # type: ignore
    })

    response = matching_runnable.invoke({
        "job_desc": job_desc,
        "resume": resume,
        "raw_resume": raw_resume
    })

    return response

In [27]:
anal_response1 = analysis_in_parallel(llm, parsing_response2["parsed_jd"], parsing_response2["parsed_resume"], parsing_response2["raw_resume"])

In [28]:
anal_response2 = analysis_in_parallel(model, parsing_response2["parsed_jd"], parsing_response2["parsed_resume"], parsing_response2["raw_resume"])

In [31]:
print(anal_response1["ats_result"])

score=82 issues=['Inclusion of sensitive personal information (Date of Birth) which can trigger bias filters or privacy flags in some ATS.', 'The social links (LinkedIn, GitHub, Portfolio) are listed as text labels without visible URLs, which may not be parsed correctly if not hyperlinked.', 'The Hobbies section contains non-professional interests that do not contribute to keyword optimization.', 'The References section is generally discouraged on a resume as it takes up valuable space that could be used for further skill or project elaboration.', 'The internship duration is very short (1 month, ongoing), which may need more context on specific deliverables to pass through experience-based filters.'] suggestions=['Remove the Date of Birth to comply with modern data privacy standards and reduce bias.', 'Ensure all profile links (GitHub, LinkedIn) are formatted as full URLs to ensure they are crawlable by all ATS versions.', 'Remove the References section and the Hobbies section to free 

In [29]:
anal_response2

{'matching_analysis': MatchingResponse(score=30, matched_skills=['Python', 'Java'], skill_gaps=['AWS services (API Gateway, Lambda, SQS)', 'Integration frameworks (MuleSoft, Apache Camel)', 'API development (OpenAPI/Swagger)', 'Node.js', 'Message queues and streaming platforms (Kafka, RabbitMQ)', 'Security protocols (OAuth 2.0, JWT)'], suggestions=['Gain experience with AWS services such as API Gateway, Lambda, and SQS.', 'Learn and get certified in integration frameworks like MuleSoft or Apache Camel.', 'Develop skills in API development using OpenAPI/Swagger.', 'Acquire knowledge in Node.js for backend development.', 'Understand and work with message queues and streaming platforms like Kafka and RabbitMQ.', 'Familiarize yourself with security protocols such as OAuth 2.0 and JWT.', 'Increase professional experience to meet the 5-year requirement for the role.']),
 'ats_result': ATSResponse(score=75, issues=['The resume lacks specific keywords from the job description.', 'The resume do